# Reproducing Dias et al. (2025) from Dryad dataset bvq83bkk9

**Paper.** Dias AC, Lemoine NP, Yanco SW, Zimova M, Bay RA, Weeks BC (2025). Long-term increases in
wing length occur independently of changes in climate and climate-driven shifts in body size.
*Proc. R. Soc. B* 292: 20242556. [doi:10.1098/rspb.2024.2556](https://doi.org/10.1098/rspb.2024.2556).
The claims file was written from the version of record (Europe PMC, PMC11750374) and its electronic
supplementary material (Tables S1 to S11, Fig. S1).

**Dataset.** [DRYAD:bvq83bkk9](https://doi.org/10.5061/dryad.bvq83bkk9), version id 333377 (Dryad's
version number 7, published 2024-12-09). The paper cites the dataset by its DOI, which always
resolves to the latest version; Dryad's API lists one version, 333377, so it is the one pinned. The
files are `Data.zip` (SHA-256 2185ac50e9a59c3a1dff59473216de2ab001d0d0044b1fa15442ae18a520d554)
and `README.md` (6c19de67b1c6c52802be42a6593c175a02bee0dab0ca2f6f958f08a0a3fa0482). Dryad needs a
token to download, so the workflow stages the files before the notebook runs and they are read in
place; the first cell checks their SHA-256.

**Outside input.** The paper's code, Zenodo
[doi:10.5281/zenodo.14226724](https://doi.org/10.5281/zenodo.14226724), `Code.zip` (SHA-256
cc9f58b9855aec4e3aca09be4138707d07fc34739ec5762e54e1f5bca48ddbc8), read in memory. It gives the
Stan model (`lm_mod.stan`), the column numbers the scripts select and the rows they drop. No data
come from outside the dataset: the phylogeny the preliminary models use, `chitree.rds`, is in it.

**Notebooks.** This notebook holds the data check, the sample sizes and every claim on the
preliminary phylogenetic mixed models (Tables S4 to S8). The hierarchical Stan models of Tables 1
to 3 are slow (see `stan_models.ipynb`, which times them, and `autumn_ahy_main_a.ipynb`, `_c` and `_d`, which fit
the smallest, once per filtering reading); the claims on them are blocked for time except where the attempt record says otherwise.

| Claim | Tested here | Status |
|---|---|---|
| sample-sizes | yes | tested |
| s4-s6-spring-migration-temperature-none | yes | tested |
| s5-s6-ahy-prebreeding-wintering-temperature-none | yes | tested |
| s6-autumn-ahy-recent-breeding-temperature-none | yes | tested |
| s4-hy-autumn-migration-temperature-none | yes | tested |
| s6-autumn-ahy-recent-wintering-prebreeding-precipitation-none | yes | tested |
| s6-autumn-ahy-recent-breeding-precipitation-none | yes | tested |
| s7-low-phylogenetic-signal | yes | tested |
| s4-s6-no-multicollinearity | yes | tested |
| s4-s6-climate-variable-selection | yes | tested |
| s4-s5-climate-size-interactions | yes | tested |
| s8-climate-trends | yes | tested |
| table3-autumn-ahy-prior-wintering-precipitation | preliminary part here, main model in `autumn_ahy_main_a.ipynb`, `_c` and `_d` | see record |
| claims on the HY and spring AHY main models (Tables 1, 2, S1), climate-trends-cannot-explain-wing-increase, climate-explains-little, s9-s11-expanded-ranges | no | blocked, too_large |
| claims on the autumn AHY main model only (Table 3) | in `autumn_ahy_main_a.ipynb`, `_c` and `_d` | see record |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import wingallom as w

warnings.filterwarnings("ignore", module="rdata")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
print("Dryad dataset", w.DRYAD_ID, "version", w.DRYAD_VERSION)
for name, digest in w.check_staged().items():
    print(f"{name}: SHA-256 {digest} (matches Dryad's file list)")
with w.data_zip() as z:
    print("\nData.zip holds:")
    for info in z.infolist():
        if "__MACOSX" not in info.filename and not info.filename.endswith(("/", ".DS_Store")):
            print(f"  {info.filename}  {info.file_size} bytes")

## Data check

Each check of the description against the values is stated under its output. The dataset has no
time series, so there is no clock to align: every table is keyed by species and year.

In [ ]:
raw = w.read_csv("morphdata.csv")
print("morphdata.csv:", raw.shape, "columns", list(raw.columns))
print("missing values per column:", raw.isna().sum().to_dict())
print("\nSpecimens by class (Season, NewAge):")
print(raw.groupby(["Season", "NewAge"]).size().to_string())
print("\nSpecimens by species and class:")
print(pd.crosstab(raw.Taxon, [raw.Season, raw.NewAge], margins=True).to_string())
print("\nSex values:", raw.Sex.value_counts().to_dict())
print("Years:", raw.Year.min(), "to", raw.Year.max(), "; species-years with specimens:",
      raw.groupby(["Taxon", "Year"]).ngroups)
print("\nTarsus and wing (mm) by species:")
print(raw.groupby("Taxon")[["Tarsus", "Wing"]].agg(["min", "median", "max"]).to_string())
print("\nsize_idx and wing_idx (not used):", raw[["size_idx", "wing_idx"]].describe().loc[["min", "max"]].round(2).to_dict())

The columns, species, sexes, ages, seasons and years (1981 to 2016) are those the README lists. No
spring specimen is HY, as the Methods say. Tarsus (19.2 to 28.0 mm) and wing (52 to 90 mm) are in
plausible ranges for these sparrows, junco and ovenbird. The file holds 27 486 specimens, 120 more
than the 27 366 of the paper; the next cells show where they go.

In [ ]:
cat = w.read_csv("catnodata.csv")
print("catnodata.csv:", cat.shape, "columns", list(cat.columns))
print("rows without a catalogue number:", int(cat.CatNo.isna().sum()))
known = cat.dropna(subset=["CatNo"])
print("catalogue numbers given more than once:", int(known.CatNo.duplicated(keep=False).sum()),
      "rows,", int(known.CatNo.duplicated().sum()), "repeats; institution codes:", cat.InstCode.value_counts(dropna=False).to_dict())
dup = known[known.CatNo.duplicated(keep=False)].sort_values("CatNo")
print("repeated catalogue numbers that differ in species, sex, age, year or season:",
      int(dup.groupby("CatNo")[["Taxon", "Sex", "Age", "Year", "Season"]].nunique().gt(1).any(axis=1).sum()))
counts = pd.concat({"morphdata": raw.groupby(["Season", "NewAge"]).size(),
                    "catnodata": cat.rename(columns={"Age": "NewAge"}).groupby(["Season", "NewAge"]).size()}, axis=1)
print("\nRows by class in the two files:")
print(counts.to_string())
print("\nmorphdata.csv has no catalogue number, so its rows cannot be matched to catnodata.csv one by one.")
same = raw.duplicated(subset=["Taxon", "Sex", "NewAge", "Year", "Season", "Tarsus", "Wing"], keep=False)
print("morphdata rows identical to another row in every recorded field:", int(same.sum()),
      f"({same.mean():.1%}); measurements are rounded to 0.1 mm (tarsus) and 0.5 mm (wing), so such ties are expected")
print("wing values that are not a multiple of 0.5 mm:", int((raw.Wing * 2 % 1 != 0).sum()),
      "; tarsus values with more than one decimal:", int(((raw.Tarsus * 10).round(6) % 1 != 0).sum()))

`catnodata.csv`, which the paper cites for catalogue numbers, has 26 436 rows, 1050 fewer than
`morphdata.csv`, and 5061 of them have no catalogue number. Catalogue numbers repeat, so the file
cannot show that each specimen was deposited once, and `morphdata.csv` has no identifier to link a
measurement to a specimen. Identical rows in `morphdata.csv` are expected from the rounding of the
measurements and are not evidence of duplicates.

In [ ]:
ed = {cls: w.read_csv(w.ED_FILE[cls]) for cls in w.CLASSES}
for cls, df in ed.items():
    print(f"{w.ED_FILE[cls]}: {df.shape}, y_id {df.y_id.min()} to {df.y_id.max()}, species {df.spec_id.nunique()}")
print("columns of ed_FHY.csv:", list(ed["HY"].columns))
print("columns of ed_AHYspring.csv not in ed_FHY.csv:", sorted(set(ed["spring AHY"].columns) - set(ed["HY"].columns)))
print("columns of ed_FHY.csv not in ed_AHYspring.csv:", sorted(set(ed["HY"].columns) - set(ed["spring AHY"].columns)))
print("\ned_FHY.csv and ed_AHYfall.csv are identical:", ed["HY"].equals(ed["autumn AHY"]))
print("ed_FHY.csv Year2 - y_id:", (ed["HY"].Year2 - ed["HY"].y_id).unique().tolist(),
      "; ed_AHYspring.csv y_id - Year:", (ed["spring AHY"].y_id - ed["spring AHY"].Year).unique().tolist())
ed_hy, ed_sp = ed["HY"], ed["spring AHY"]
both = ed_hy.merge(ed_sp, left_on=["spec_id", "y_id"], right_on=["spec_id", "Year"], suffixes=("_f", "_s"))
climate_cols = [c for c in ed_sp.columns if c not in ("Unnamed: 0", "y_id", "spec_id", "Species2", "Year")]
diff = max(np.nanmax(np.abs(both[c + "_f"] - both[c + "_s"])) for c in climate_cols if c + "_f" in both)
print(f"ed_AHYspring.csv row for Year Y against ed_FHY.csv row for y_id Y, {len(both)} species-years: "
      f"largest difference over the shared climate columns {diff}")
lag = ed_hy.merge(ed_hy, left_on=["spec_id", "y_id"], right_on=["spec_id", "Year2"], suffixes=("", "_prev"))
pairs = [("PrevTempWinMo", "TempWinMo"), ("PrevPrecipWinMo", "PrecipWinMo"), ("PrevTempBreedBase", "TempBreedMoBase"),
         ("PrevPrecipBreedBase", "PrecipBreedMoBase"), ("PrevMinTempFallMig", "MinTempFallMig"),
         ("PrevAvgTempPreBreedBase", "AvgTempPreBreedBase"), ("PrevMinTempSpringMig", "MinTempSpringMig")]
print("'Prev' columns against the same variable one year earlier (largest difference):",
      {p: float(np.nanmax(np.abs(lag[p] - lag[c + "_prev"]))) for p, c in pairs})
values = ed_hy[[c for c in climate_cols if c in ed_hy.columns]]
per_species = ed_hy.groupby("spec_id")[["TempBreedMoBase", "PrecipBreedMoBase", "MinTempFallMig", "PrecipWinMo"]].agg(["mean", "std"])
print("\nClimate values (ed_FHY.csv), all columns: min", round(float(np.nanmin(values)), 2), "max", round(float(np.nanmax(values)), 2))
print("Mean and s.d. within species of four columns (README: scaled to mean 0 and s.d. 1 within species):")
print(per_species.round(3).to_string())
print("\nMissing values per climate column used by the models (ed_FHY.csv):")
print(ed_hy[["PrevTempWinMo", "PrevPrecipWinMo", "AvgTempPreBreedBase", "PrecipPreBreedBase", "TempBreedMoBase",
         "PrecipBreedMoBase", "MinTempFallMig", "MinTempSpringMig", "PrevPrevPrecipWinMo", "PrevTempBreedBase"]].isna().sum().to_dict())

The README says the climate is scaled to mean 0 and s.d. 1 within species. Over the years in the
files the means are near 0 and the s.d. near 1, but not exactly; this is consistent with scaling
over a longer span (the Methods give 1980 to 2017), not shown. The README says `ed_FHY.csv` and
`ed_AHYfall.csv` hold the climate of HY and autumn AHY birds; the two files are identical, which
fits, since both classes are collected in the autumn of `y_id` and the autumn AHY model uses the
"Prev" columns. The spring file's `Year` is `y_id - 1`, as the README says, and its rows equal the
`ed_FHY.csv` rows of the same year, so the three files are one climate table. The "Prev" columns
are the variable of the year before.

In [ ]:
tree = w.read_tree()
print("chitree.rds: a phylo object with", len(tree["tip_label"]), "tips and", tree["Nnode"], "internal nodes")
print("the five species are tips:", all(s in tree["tip_label"] for s in w.SPECIES))
cov = w.tip_covariance(tree, w.SPECIES)
print("\nShared branch length from the root (vcv.phylo of the five tips, millions of years as in BirdTree):")
print(cov.round(2).to_string())
print("ultrametric over the five tips (equal root-to-tip lengths):", bool(np.allclose(np.diag(cov), cov.iloc[0, 0])))

## Data preparation and sample sizes

The scripts select climate columns by their number in R's merged table and drop rows by
hard-coded number. `wingallom.prepare` rebuilds the merged table, selects by the same numbers,
finds the rows with a missing value and drops those; the cell below shows that they are the rows
the code drops, and which columns the numbers select.

In [ ]:
READINGS = {"a": "as deposited, no filter", "b": "5 MAD per species, R mad()", "c": "5 MAD per species and sex",
            "d": "5 MAD per species, unscaled MAD"}
removed = {r: w.outlier_mask(raw, r) for r in READINGS}
print("Specimens the outlier filter removes, by reading:")
for r, mask in removed.items():
    by = raw[mask].groupby(["Season", "NewAge"]).size().to_dict()
    print(f"  ({r}) {READINGS[r]}: {int(mask.sum())}  by class {by}")
print("readings (a) and (b) coincide:", bool((removed['a'] == removed['b']).all()))
RUN = ["a", "c", "d"]   # (b) removes nothing, so it is (a)

morph = {r: w.morph_prepared(r) for r in RUN}
prep = {(r, which, cls): w.prepare(morph[r], cls, which) for r in RUN for which in ("main", "prelim") for cls in w.CLASSES}
for which in ("main", "prelim"):
    for cls in w.CLASSES:
        p = prep[("a", which, cls)]
        print(f"\n{which} {cls}: {p['n_merged']} species-years with specimens; rows with a missing value {p['missing_rows']};"
              f" rows the code drops {p['dropped_rows']}; same: {p['missing_rows'] == p['dropped_rows']}")
        print("  selected columns:", p["selected"])

In [ ]:
expected = {("all", "specimens"): 27366,
            ("main", "HY", "species-years"): 177, ("main", "HY", "birds"): 11798,
            ("main", "spring AHY", "species-years"): 174, ("main", "spring AHY", "birds"): 12318,
            ("main", "autumn AHY", "species-years"): 171, ("main", "autumn AHY", "birds"): 3250,
            ("prelim", "HY", "birds"): 11731, ("prelim", "spring AHY", "birds"): 11481,
            ("prelim", "autumn AHY", "birds"): 3122}
rows = []
for key, value in expected.items():
    row = {"quantity": " ".join(key), "paper": value, "tolerance": round(value * 0.1)}
    for r in RUN:
        if key[0] == "all":
            found = len(morph[r])
        else:
            p = prep[(r, key[0], key[1])]
            found = len(p["G"]) if key[2] == "species-years" else len(p["md"])
        row[f"({r})"] = found
    rows.append(row)
sizes = pd.DataFrame(rows)
for r in RUN:
    sizes[f"within ({r})"] = (sizes[f"({r})"] - sizes.paper).abs() <= sizes.tolerance
print(sizes.to_string(index=False))
modelled = sum(len(prep[("a", "main", c)]["md"]) for c in w.CLASSES)
print("\nSpecimens in the three main models together, reading (a):", modelled,
      "; specimens in species-years dropped for missing climate:", len(morph["a"]) - modelled)
print("Main-model specimens by species, reading (a):")
print(pd.DataFrame({c: prep[("a", "main", c)]["md"].Taxon.value_counts() for c in w.CLASSES}).to_string())

**Result: reproduced.** As deposited (reading a), every count equals the paper's: 177, 174 and 171
species-years and 11 798, 12 318 and 3250 birds in the main models, and 11 731, 11 481 and 3122 birds
in the preliminary models. The three main models together hold 27 366 birds, the paper's total; the
file's 27 486 include 120 birds in species-years dropped for a missing climate value. Readings (c)
and (d) remove 17 and 70 specimens and every count stays within tolerance, but only reading (a)
gives the paper's counts exactly. The rows the code drops by number are the rows with a missing
value in the columns it selects, for all six tables.

## Preliminary phylogenetic mixed models (Tables S4 to S7)

Script02 fits, per class, scaled wing on scaled tarsus, sex, every climate variable and year, each
with its tarsus interaction, with a random effect of species whose covariance is the phylogeny;
inverse-Wishart priors V = 1, nu = 0.02; 4 chains of 15 000 iterations with 5000 burn-in (thinning
10, MCMCglmm's default). MCMCglmm is an R package and cannot be installed in this environment;
`wingallom.gibbs_glmm` samples the same model with the same Gibbs scheme. The code sets no seed;
the seeds here are a standard choice, one generator per class, reading and chain. The models are
fitted under filtering readings (a), (c) and (d), as the claims require.

Interval readings, as the claims file names them: (i) the 2.5 and 97.5% quantiles of the four
pooled chains; (ii) the 95% HPD interval of chain 4. pMCMC is from chain 4.

In [ ]:
Ainv = w.scaled_tip_inverse(tree, w.SPECIES)
print("Inverse of the scaled tip covariance (inverseA, scale = TRUE), species in order", w.SPECIES)
print(np.round(Ainv, 3))
prelim = {}
t0 = time.time()
for ri, r in enumerate(RUN):
    for ci, cls in enumerate(w.CLASSES):
        X, names, y, groups = w.prelim_design(prep[(r, "prelim", cls)], w.PRELIM_TERMS[cls])
        chains = [w.gibbs_glmm(X, y, groups, w.SPECIES, Ainv, np.random.default_rng([2025, ri, ci, k])) for k in range(4)]
        lam = chains[3]["VCV"][:, 0] / chains[3]["VCV"].sum(axis=1)
        prelim[(r, cls)] = {"summary": w.glmm_summary(chains, names), "vif": w.vif(X, names), "lambda": lam,
                            "n": len(y), "p": X.shape[1], "chains": chains, "names": names}
print(f"\n{len(prelim)} models of 4 chains in {time.time() - t0:.0f} s")
print(pd.DataFrame({k: {"birds": v["n"], "fixed effects": v["p"], "kept draws per chain": len(v["chains"][0]["Sol"])}
                    for k, v in prelim.items()}).T.to_string())

In [ ]:
def gelman_rubin(chains, k):
    x = np.array([c["Sol"][:, k] for c in chains])
    n = x.shape[1]
    wv = x.var(axis=1, ddof=1).mean()
    b = n * x.mean(axis=1).var(ddof=1)
    return np.sqrt(((n - 1) / n * wv + b / n) / wv)

for (r, cls), v in prelim.items():
    psrf = [gelman_rubin(v["chains"], k) for k in range(v["p"])]
    print(f"({r}) {cls}: largest Gelman-Rubin factor over the fixed effects {max(psrf):.3f}")
for cls in w.CLASSES:
    print(f"\nTable S{4 + w.CLASSES.index(cls)}, {cls}, reading (a):")
    print(prelim[("a", cls)]["summary"].round(3).to_string())

### Helper for the claims of absence

For a list of climate terms, each claim below prints, per filtering reading, the estimate and both
intervals of each main effect and tarsus interaction, and whether any interval excludes 0.

In [ ]:
def absence(cls, terms, label):
    rows = []
    for r in RUN:
        s = prelim[(r, cls)]["summary"]
        for t in terms:
            for name in (t, f"scaleTarsus:{t}"):
                q = s.loc[name]
                rows.append({"reading": r, "term": name, "mean": q.mean_pooled, "q2.5": q["q2.5_pooled"],
                             "q97.5": q["q97.5_pooled"], "hpd_lo_c4": q.hpd_lo_c4, "hpd_hi_c4": q.hpd_hi_c4,
                             "pMCMC_c4": q.pMCMC_c4, "excludes 0 (i)": q.excl0_i, "excludes 0 (ii)": q.excl0_ii})
    out = pd.DataFrame(rows)
    print(f"{label} ({cls} preliminary model):")
    print(out.round(4).to_string(index=False))
    print("any interval excluding 0, by reading:",
          out.groupby("reading")[["excludes 0 (i)", "excludes 0 (ii)"]].any().to_dict("index"))
    return out

### Monte Carlo check for intervals that end near 0

The code sets no seed, so a 95% bound that ends within a few thousandths of 0 can fall on either
side with another seed. `seed_check` refits one model with ten other seed sets and counts how often
each interval of one term excludes 0. It is used below where a bound is that close.

In [ ]:
def seed_check(r, cls, term, n_sets=10):
    X, names, y, groups = w.prelim_design(prep[(r, "prelim", cls)], w.PRELIM_TERMS[cls])
    k = names.index(term)
    rows = []
    for seed in range(n_sets):
        chains = [w.gibbs_glmm(X, y, groups, w.SPECIES, Ainv, np.random.default_rng([99, seed, j])) for j in range(4)]
        pooled = np.concatenate([c["Sol"][:, k] for c in chains])
        q = np.quantile(pooled, [0.025, 0.975])
        lo, hi = w.hpd(chains[3]["Sol"][:, k])
        rows.append({"seed set": seed, "q2.5": q[0], "q97.5": q[1], "hpd_lo_c4": lo, "hpd_hi_c4": hi,
                     "excludes 0 (i)": q[0] > 0 or q[1] < 0, "excludes 0 (ii)": lo > 0 or hi < 0})
    out = pd.DataFrame(rows)
    print(f"{term}, {cls}, reading ({r}), {n_sets} other seed sets:")
    print(out.round(4).to_string(index=False))
    print(f"interval (i) excludes 0 in {int(out['excludes 0 (i)'].sum())} of {n_sets}; "
          f"interval (ii) in {int(out['excludes 0 (ii)'].sum())} of {n_sets}")
    return out

## s4-s6-spring-migration-temperature-none

> Spring migration temperature has no significant effect on wing length, or interaction with body
> size, in any of the three preliminary phylogenetic mixed models, and so is left out of the main
> models.

Terms: MinTempSpringMig (all three models), MinTempNextSpringMig (spring AHY) and
PrevMinTempSpringMig (autumn AHY), with their tarsus interactions.

In [ ]:
spring_mig = pd.concat([
    absence("HY", ["MinTempSpringMig"], "Spring migration temperature"),
    absence("spring AHY", ["MinTempSpringMig", "MinTempNextSpringMig"], "Spring migration temperature"),
    absence("autumn AHY", ["PrevMinTempSpringMig", "MinTempSpringMig"], "Spring migration temperature")])
print("\nAny of the", len(spring_mig) // len(RUN), "intervals per reading excludes 0:",
      spring_mig.groupby("reading")[["excludes 0 (i)", "excludes 0 (ii)"]].any().to_dict("index"))

In [ ]:
_ = seed_check("a", "spring AHY", "MinTempNextSpringMig")

**Result: not reproduced, on a bound within Monte Carlo error of 0.** Every interval of the spring
migration temperatures and their tarsus interactions includes 0 under interval reading (i), in all
three models and filtering readings. Under reading (ii), the chain-4 HPD interval of
MinTempNextSpringMig in the spring AHY model ends at -0.0002 (reading a) and -0.0005 (reading c),
excluding 0; its pooled interval is -0.0275 to 0.0007, pMCMC 0.056. The criterion counts any
exclusion as not reproduced. With ten other seed sets the same HPD bound lies between -0.0000 and
0.0022 and excludes 0 in 1 of 10, so the call rests on the Monte Carlo draw; the paper prints this
bound as 0.00.

## s5-s6-ahy-prebreeding-wintering-temperature-none

> Pre-breeding and wintering season temperatures have no significant association with AHY wing
> length in the preliminary models of spring and autumn AHY birds.

In [ ]:
ahy_temp = pd.concat([
    absence("spring AHY", ["TempWinMo", "PrevTempWinMo", "AvgTempPreBreedBase"], "Wintering and pre-breeding temperature"),
    absence("autumn AHY", ["PrevTempWinMo", "PrevPrevTempWinMo", "AvgTempPreBreedBase", "PrevAvgTempPreBreedBase"],
            "Wintering and pre-breeding temperature")])
print("\nAny interval excludes 0:", ahy_temp.groupby("reading")[["excludes 0 (i)", "excludes 0 (ii)"]].any().to_dict("index"))
print("Terms whose interval excludes 0:")
print(ahy_temp[ahy_temp["excludes 0 (i)"] | ahy_temp["excludes 0 (ii)"]][["reading", "term", "mean", "q2.5", "q97.5", "hpd_lo_c4", "hpd_hi_c4"]].round(4).to_string(index=False))

**Result: reproduced.** In the spring and autumn AHY preliminary models, every interval of the
wintering and pre-breeding temperatures (TempWinMo, PrevTempWinMo, PrevPrevTempWinMo,
AvgTempPreBreedBase, PrevAvgTempPreBreedBase) and their tarsus interactions includes 0, under both
interval readings and all three filtering readings.

## s6-autumn-ahy-recent-breeding-temperature-none

> In the autumn AHY preliminary model, temperature of the most recent breeding season is not
> significantly associated with wing length.

In [ ]:
_ = absence("autumn AHY", ["TempBreedMoBase"], "Most recent breeding temperature")

**Result: reproduced.** TempBreedMoBase (0.022, -0.017 to 0.061 under reading a) and its tarsus
interaction include 0 under both interval readings and all filtering readings.

## s4-hy-autumn-migration-temperature-none

> In the HY preliminary model, autumn migration temperature has no significant association with
> wing length.

In [ ]:
_ = absence("HY", ["MinTempFallMig"], "Autumn migration temperature")

**Result: reproduced.** In the HY model, MinTempFallMig (-0.007, -0.020 to 0.006 under reading a) and
its tarsus interaction include 0 under both interval readings and all filtering readings.

## s6-autumn-ahy-recent-wintering-prebreeding-precipitation-none

> In the autumn AHY preliminary model, precipitation of the most recent wintering season and
> pre-breeding period is not associated with wing length.

Terms as the criterion gives them: PrevPrecipWinMo and PrecipPreBreedBase.

In [ ]:
_ = absence("autumn AHY", ["PrevPrecipWinMo", "PrecipPreBreedBase"], "Most recent wintering and pre-breeding precipitation")

In [ ]:
_ = seed_check("d", "autumn AHY", "scaleTarsus:PrevPrecipWinMo")

**Result: not reproduced, on a bound within Monte Carlo error of 0.** PrevPrecipWinMo,
PrecipPreBreedBase and their tarsus interactions include 0 under both interval readings in
filtering readings (a) and (c), and under interval reading (i) in (d). Under reading (d), the chain-4
HPD interval of the tarsus by PrevPrecipWinMo interaction is -0.0532 to -0.0003 and excludes 0; the
criterion counts this as not reproduced. With ten other seed sets that bound lies between -0.0013
and 0.0040 and excludes 0 in 1 of 10.

## s6-autumn-ahy-recent-breeding-precipitation-none

> In the autumn AHY preliminary model, precipitation of the most recent breeding season is not
> significant.

In [ ]:
_ = absence("autumn AHY", ["PrecipBreedMoBase"], "Most recent breeding precipitation")

**Result: reproduced.** PrecipBreedMoBase (0.020, -0.018 to 0.059 under reading a) and its tarsus
interaction include 0 under both interval readings and all filtering readings.

## table3-autumn-ahy-prior-wintering-precipitation, preliminary part

The claim also needs the autumn AHY main model (`autumn_ahy_main_a.ipynb`, `_c` and `_d`). Here: PrevPrecipPreBreedBase
and its tarsus interaction in the autumn AHY preliminary model (Table S6 reports -0.00 to 0.06).

In [ ]:
_ = absence("autumn AHY", ["PrevPrecipPreBreedBase"], "Pre-breeding precipitation before development")

Preliminary part: PrevPrecipPreBreedBase (0.027, -0.002 to 0.055 under reading a; the paper -0.00 to
0.06) and its tarsus interaction include 0 under both interval readings and all filtering readings.
The main-model part and the outcome are in `autumn_ahy_main_a.ipynb`, `_c` and `_d`.

## s7-low-phylogenetic-signal

> Phylogenetic signal is low in all three preliminary models (posterior mode of lambda 0.02, 0.02
> and 0.04), so the main models do not control for phylogeny.

Lambda from chain 4, species variance over the sum of species and residual variances; posterior
mode (MCMCglmm's `posterior.mode`) and 95% HPD. Reported HPDs: HY 0.01 to 0.21, spring AHY 0.01 to
0.16, autumn AHY 0.01 to 0.21.

In [ ]:
reported = {"HY": (0.02, 0.01, 0.21), "spring AHY": (0.02, 0.01, 0.16), "autumn AHY": (0.04, 0.01, 0.21)}
rows = []
for r in RUN:
    for cls in w.CLASSES:
        lam = prelim[(r, cls)]["lambda"]
        lo, hi = w.hpd(lam)
        mode = w.posterior_mode(lam)
        pm, plo, phi = reported[cls]
        rows.append({"reading": r, "class": cls, "mode": mode, "mean": lam.mean(), "hpd_lo": lo, "hpd_hi": hi,
                     "paper mode": pm, "paper HPD": f"{plo} to {phi}", "mode in paper HPD": plo <= mode <= phi})
lam_table = pd.DataFrame(rows)
print(lam_table.round(3).to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, cls in zip(axes, w.CLASSES):
    lam = prelim[("a", cls)]["lambda"]
    ax.hist(lam, bins=50, range=(0, 1))
    ax.set_title(f"{cls}, reading (a)")
    ax.set_xlabel("lambda (chain 4)")
    print(f"{cls}: lambda draws range {lam.min():.4f} to {lam.max():.4f}; all inside the 0 to 1 axis")
plt.tight_layout()
plt.show()

The mode of a posterior that piles up against 0 depends on the chain. As a check of that spread,
the lambda mode of each of the four chains, reading (a):

In [ ]:
for cls in w.CLASSES:
    modes = [w.posterior_mode(c["VCV"][:, 0] / c["VCV"].sum(axis=1)) for c in prelim[("a", cls)]["chains"]]
    print(cls, "modes of chains 1 to 4:", np.round(modes, 3))

**Result: reproduced.** The posterior modes of lambda are 0.056, 0.062 and 0.087 (HY, spring AHY,
autumn AHY; reading a), and 0.038 to 0.089 over all readings, each inside the paper's HPD interval
(0.01 to 0.21, 0.01 to 0.16, 0.01 to 0.21). The modes are larger than the paper's 0.02, 0.02 and
0.04, and the HPD intervals found here are wider (upper bounds 0.30 to 0.48); the modes of single
chains range from 0.041 to 0.096. Lambda is low by the paper's own measure, which names no level for
"low".

## s4-s6-no-multicollinearity

> The predictors of the preliminary models are not collinear; every variance inflation factor is
> below 10 (largest 3.34 for HY, 2.43 for spring AHY, 6.08 for autumn AHY).

In [ ]:
paper_vif = {"HY": 3.34, "spring AHY": 2.43, "autumn AHY": 6.08}
rows = []
for r in RUN:
    for cls in w.CLASSES:
        v = prelim[(r, cls)]["vif"]
        rows.append({"reading": r, "class": cls, "terms": len(v), "largest VIF": v.max(), "term": v.idxmax(),
                     "all below 10": bool((v < 10).all()), "paper": paper_vif[cls],
                     "within tolerance": abs(v.max() - paper_vif[cls]) <= round(paper_vif[cls] * 0.1, 2)})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nVIFs, reading (a):")
print(pd.DataFrame({cls: prelim[("a", cls)]["vif"] for cls in w.CLASSES}).round(2).to_string())

**Result: reproduced.** Every VIF is below 10. The largest are 3.339, 2.427 and 6.084 under reading
(a) (paper 3.34, 2.43 and 6.08), and 3.312 to 3.379, 2.400 to 2.427 and 6.084 to 6.472 over the
three filtering readings, all within tolerance.

## s4-s6-climate-variable-selection

> The climate variables with 95% intervals excluding 0 in each preliminary model are those
> included in that class's main model.

In [ ]:
rows = []
for r in RUN:
    for cls in w.CLASSES:
        s = prelim[(r, cls)]["summary"]
        climate = [t for t in w.PRELIM_TERMS[cls] if t != "y_id"]
        main = set(w.MAIN_PREDICTORS[cls]) - {"y_id"}
        for reading in ("i", "ii"):
            sig = {t for t in climate if s.loc[t, f"excl0_{reading}"]}
            rows.append({"reading": r, "interval": reading, "class": cls, "selected": sorted(sig),
                         "missing from selection": sorted(main - sig), "extra in selection": sorted(sig - main),
                         "differences": len(main ^ sig)})
sel = pd.DataFrame(rows)
print(sel[["reading", "interval", "class", "missing from selection", "extra in selection", "differences"]].to_string(index=False))
print("\nDifferences summed over the three classes:")
print(sel.groupby(["reading", "interval"]).differences.sum().to_string())
print("\nBounds near 0 that the claim names (unrounded), reading (a):")
near = [("HY", "MinTempSpringMig"), ("spring AHY", "PrecipWinMo"), ("autumn AHY", "PrevPrevPrecipWinMo"), ("autumn AHY", "MinTempFallMig")]
print(pd.DataFrame([{"class": c, "term": t, **prelim[("a", c)]["summary"].loc[t, ["mean_pooled", "q2.5_pooled", "q97.5_pooled", "hpd_lo_c4", "hpd_hi_c4"]].to_dict()}
                    for c, t in near]).round(4).to_string(index=False))

**Result: reproduced.** Under interval reading (i) the climate main effects whose interval excludes
0 are exactly the main models' climate predictors in all three classes, under every filtering
reading (0 differences). Under reading (ii) MinTempNextSpringMig is also selected in the spring AHY
model under filtering readings (a) and (c) (its HPD bound is the one discussed above), one
difference.

## s4-s5-climate-size-interactions

> In the preliminary models, previous wintering temperature interacts positively with body size in
> HY birds, previous wintering precipitation interacts positively with body size in spring AHY
> birds, and no climate variable interacts with body size in autumn AHY birds.

Tarsus by climate interactions only (not sex or year).

In [ ]:
named = {"HY": "scaleTarsus:PrevTempWinMo", "spring AHY": "scaleTarsus:PrevPrecipWinMo"}
rows = []
for r in RUN:
    for reading in ("i", "ii"):
        for cls in w.CLASSES:
            s = prelim[(r, cls)]["summary"]
            inter = [f"scaleTarsus:{t}" for t in w.PRELIM_TERMS[cls] if t != "y_id"]
            sig = [t for t in inter if s.loc[t, f"excl0_{reading}"]]
            rows.append({"reading": r, "interval": reading, "class": cls,
                         "named": named.get(cls, "-"),
                         "named positive and excludes 0": (s.loc[named[cls], f"excl0_{reading}"] and s.loc[named[cls], "mean_pooled"] > 0) if cls in named else None,
                         "other interactions excluding 0": [t for t in sig if t != named.get(cls)]})
inter_table = pd.DataFrame(rows)
print(inter_table.to_string(index=False))
print("\nThe named interactions and the interactions near the line (unrounded), all readings:")
show = [("HY", "scaleTarsus:PrevTempWinMo"), ("spring AHY", "scaleTarsus:PrevPrecipWinMo"), ("HY", "scaleTarsus:AvgTempPreBreedBase"),
        ("autumn AHY", "scaleTarsus:MinTempFallMig")]
print(pd.DataFrame([{"reading": r, "class": c, "term": t, **prelim[(r, c)]["summary"].loc[t, ["mean_pooled", "q2.5_pooled", "q97.5_pooled", "hpd_lo_c4", "hpd_hi_c4", "pMCMC_c4"]].to_dict()}
                    for r in RUN for c, t in show]).round(4).to_string(index=False))

The HY tarsus by pre-breeding temperature interval ends near 0; the same check:

In [ ]:
_ = seed_check("a", "HY", "scaleTarsus:AvgTempPreBreedBase")

**Result: reproduced.** Under interval reading (i), and every filtering reading, the HY tarsus by
PrevTempWinMo (0.0155, 0.0030 to 0.0277 under reading a) and the spring AHY tarsus by PrevPrecipWinMo
(0.0151, 0.0003 to 0.0295) interactions are positive and exclude 0, and no other tarsus by climate
interaction excludes 0 in any of the three models. Under reading (ii) the same holds except under
filtering reading (d), where the autumn AHY tarsus by PrevPrecipWinMo HPD also excludes 0. The HY
tarsus by pre-breeding temperature interaction lies at the line: its pooled interval ends at 0.0008
here and at 0.0002 to 0.0010 with ten other seed sets (never excluding 0), while its chain-4 HPD
interval excludes 0 in 4 of 10.

## s8-climate-trends

> Over time, pre-breeding precipitation, breeding season temperature and precipitation, autumn
> migration temperature and wintering temperature increase, wintering precipitation decreases, and
> pre-breeding and spring migration temperatures show no trend.

Each variable modelled on Time = Year - 1980 with a random effect of species with the phylogenetic
covariance, priors and chains as in the preliminary models. The code's table (`envdata3`) is defined
nowhere; the claims file's readings are (a) one row per species and year of the deposited climate
data, and (b) only the species-years that have specimens. For (a), the same-year columns of
`ed_FHY.csv` give 1981 to 2016 and the spring file's rows (its `Year`) add 1980; the files hold no
same-year value for 2017. The check above showed the files agree where they overlap.

In [ ]:
same_year = pd.concat([ed_hy.assign(Year=ed_hy.y_id)[["spec_id", "Year", *w.TREND_VARIABLES]],
                       ed_sp[~ed_sp.Year.isin(ed_hy.y_id)][["spec_id", "Year", *w.TREND_VARIABLES]]]).sort_values(["spec_id", "Year"])
with_specimens = raw.groupby(["Taxon", "Year"]).size().rename("n").reset_index()
tables = {"a": same_year,
          "b": same_year.merge(with_specimens, left_on=["spec_id", "Year"], right_on=["Taxon", "Year"])}
for k, t in tables.items():
    print(f"reading ({k}): {len(t)} species-years, years {t.Year.min()} to {t.Year.max()}")
expected_trend = {"AvgTempPreBreedBase": (0.0, -0.01, 0.01, "none"), "PrecipPreBreedBase": (0.02, 0.01, 0.04, "+"),
                  "TempBreedMoBase": (0.03, 0.02, 0.04, "+"), "PrecipBreedMoBase": (0.05, 0.04, 0.06, "+"),
                  "MinTempFallMig": (0.02, 0.00, 0.03, "+"), "TempWinMo": (0.02, 0.01, 0.03, "+"),
                  "PrecipWinMo": (-0.02, -0.03, -0.01, "-"), "MinTempSpringMig": (0.0, -0.01, 0.01, "none")}
tolerance = {"PrecipPreBreedBase": 0.015, "TempBreedMoBase": 0.01, "PrecipBreedMoBase": 0.01, "MinTempFallMig": 0.015,
             "TempWinMo": 0.01, "PrecipWinMo": 0.01}
rows = []
for ti, (k, t) in enumerate(tables.items()):
    for vi, var in enumerate(w.TREND_VARIABLES):
        d = t.dropna(subset=[var])
        X = np.column_stack([np.ones(len(d)), d.Year - 1980])
        chains = [w.gibbs_glmm(X, d[var].to_numpy(float), d.spec_id.to_numpy(), w.SPECIES, Ainv,
                               np.random.default_rng([8, ti, vi, c])) for c in range(4)]
        sm = w.glmm_summary(chains, ["(Intercept)", "Time"]).loc["Time"]
        pm, plo, phi, sign = expected_trend[var]
        rows.append({"reading": k, "variable": var, "n": len(d), "mean": sm.mean_pooled, "q2.5": sm["q2.5_pooled"],
                     "q97.5": sm["q97.5_pooled"], "mean_c4": sm.mean_c4, "hpd_lo_c4": sm.hpd_lo_c4, "hpd_hi_c4": sm.hpd_hi_c4,
                     "pMCMC_c4": sm.pMCMC_c4, "paper": f"{pm} ({plo}, {phi})", "paper trend": sign})
trends = pd.DataFrame(rows)

def call(lo, hi):
    return "+" if lo > 0 else "-" if hi < 0 else "none"

trends["trend (i)"] = [call(a, b) for a, b in zip(trends["q2.5"], trends["q97.5"])]
trends["trend (ii)"] = [call(a, b) for a, b in zip(trends.hpd_lo_c4, trends.hpd_hi_c4)]
trends["mean within tolerance"] = [abs(m - expected_trend[v][0]) <= tolerance[v] if v in tolerance else None
                                   for m, v in zip(trends["mean"], trends.variable)]
print(trends.round(4).to_string(index=False))
for k in tables:
    for reading in ("i", "ii"):
        sub = trends[trends.reading == k]
        agree = (sub[f"trend ({reading})"] == sub["paper trend"]).sum()
        means = sub["mean within tolerance"].dropna()
        print(f"data ({k}), interval ({reading}): {agree} of 8 trend calls agree with the paper; "
              f"{int(means.sum())} of {len(means)} means within tolerance")

**Result: reproduced.** Under both data readings and both interval readings, pre-breeding
precipitation, breeding temperature, breeding precipitation, autumn migration temperature and
wintering temperature increase, wintering precipitation decreases, and pre-breeding and spring
migration temperatures have intervals that include 0: 8 of 8 trend calls agree with Table S8. The
six trends are 0.021, 0.034, 0.055, 0.016, 0.024 and -0.020 per year under data reading (a), all
within tolerance.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| sample-sizes | reproduced |
| s4-s6-spring-migration-temperature-none | not reproduced (one HPD bound at -0.0002, within Monte Carlo error) |
| s5-s6-ahy-prebreeding-wintering-temperature-none | reproduced |
| s6-autumn-ahy-recent-breeding-temperature-none | reproduced |
| s4-hy-autumn-migration-temperature-none | reproduced |
| s6-autumn-ahy-recent-wintering-prebreeding-precipitation-none | not reproduced (one HPD bound at -0.0003 under one filtering reading) |
| s6-autumn-ahy-recent-breeding-precipitation-none | reproduced |
| s7-low-phylogenetic-signal | reproduced |
| s4-s6-no-multicollinearity | reproduced |
| s4-s6-climate-variable-selection | reproduced |
| s4-s5-climate-size-interactions | reproduced |
| s8-climate-trends | reproduced |
| table3-autumn-ahy-prior-breeding-temperature | reproduced (autumn AHY notebooks) |
| table3-autumn-ahy-prior-autumn-temperature | reproduced (autumn AHY notebooks) |
| table3-autumn-ahy-recent-autumn-temperature | partially reproduced (autumn AHY notebooks) |
| table3-autumn-ahy-prior-wintering-precipitation | partially reproduced (autumn AHY notebooks and here) |
| the 16 claims on the HY and spring AHY Stan models, the expanded ranges and the claims that depend on them | blocked, too large (`stan_models.ipynb`) |

How the analysis differs from the paper's:

- The preliminary models are sampled by `wingallom.gibbs_glmm`, a Gibbs sampler of the same model
  and priors as MCMCglmm (an R package that cannot be installed in this Python environment), with
  seeds chosen here; the paper's runs set no seed.
- The Stan model is the paper's `lm_mod.stan` in current array syntax, run with CmdStan 2.40 in
  place of rstan; with the same seeds the draws still differ.
- The outlier filter of the Methods is not in the code; readings (a) to (d) of the claims file are
  run, (b) being identical to (a).
- The Table S8 data table is not defined in the code; it is built here from the deposited climate
  files under the claims file's two readings.

What was found about the dataset: the deposited specimens are already filtered, or were never
filtered (reading a gives every count of the paper exactly), and the README's statement that the
filtering steps are in the code does not hold; the 120 specimens beyond the paper's 27 366 sit in
species-years without climate data. `catnodata.csv` has 1050 fewer rows than `morphdata.csv`, 5061
rows without a catalogue number and 945 repeated catalogue numbers, and `morphdata.csv` has no
identifier, so measurements cannot be traced to specimens. `ed_FHY.csv` and `ed_AHYfall.csv` are
identical, and the three climate files are one table shifted by a year, consistent with the README.
The code depends on R's column order and on row numbers fixed by hand, which this notebook confirms
for the deposited files.